NoteBook 02 - Explore and Clean Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import glob
import re


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation"
FINAL_PROJECT_PATH = PROJECT_PATH + "/Final_Project"

PROCESSED_DATA_PATH = FINAL_PROJECT_PATH + "/02_Data/Processed"
COMBINED_RAW_PATH = PROCESSED_DATA_PATH + "/Combined_Raw"
CLEAN_DATA_PATH = FINAL_PROJECT_PATH + "/02_Data/Cleaned"
RESULTS_PATH = FINAL_PROJECT_PATH + "/04_Results"

os.makedirs(CLEAN_DATA_PATH, exist_ok=True)

combined_files = glob.glob(os.path.join(COMBINED_RAW_PATH, "*.csv"))
print("Combined stock files found:", len(combined_files))


Combined stock files found: 10


In [ ]:
stock_data = {}

for file_path in combined_files:
    stock_name = os.path.basename(file_path).replace("_All_Raw.csv", "")
    stock_data[stock_name] = pd.read_csv(file_path)

print("Stocks loaded:", list(stock_data.keys()))


Stocks loaded: ['Eicher_Motors', 'HDFCB', 'Mahindra_and_Mahindra', 'ICICIBC', 'ONGC', 'Hindustan_Unilever', 'Reliance', 'ITC', 'INFOSYS', 'TCS']


Timestamp conversion


In [ ]:
def parse_bloomberg_time(df):
    parsed_times = []
    row_types = []
    current_date = None

    for value in df["Time Interval"]:
        text = str(value).strip()

        if text == "Summary":
            parsed_times.append(pd.NaT)
            row_types.append("Summary")

        elif re.fullmatch(r"\d{2}[A-Za-z]{3}\d{4}_\d{2}:\d{2}:\d{2}\.\d+", text):
            # e.g. 01JAN2016_00:00:00.000000 -> remember this date for the rows that follow
            date_part = text.split("_")[0]
            current_date = pd.to_datetime(date_part, format="%d%b%Y", errors="coerce")
            parsed_times.append(pd.NaT)
            row_types.append("Date_Header")

        elif re.fullmatch(r"\d{2}:\d{2}\s*-\s*\d{2}:\d{2}", text):
            # e.g. 07:30 - 07:45 -> combine with the date we're currently tracking
            start_time = text.split("-")[0].strip()
            if current_date is not None:
                full_datetime = pd.to_datetime(
                    current_date.strftime("%Y-%m-%d") + " " + start_time,
                    format="%Y-%m-%d %H:%M", errors="coerce"
                )
                parsed_times.append(full_datetime)
                row_types.append("Interval")
            else:
                # a time row appeared before we ever saw a date header - shouldn't happen, but flagging it if it does
                parsed_times.append(pd.NaT)
                row_types.append("Interval_No_Date")

        else:
            parsed_times.append(pd.NaT)
            row_types.append("Other")

    df["datetime"] = parsed_times
    df["Row_Type"] = row_types
    return df


for stock, df in stock_data.items():
    stock_data[stock] = parse_bloomberg_time(df)

print("Timestamps parsed for all stocks.")


Timestamps parsed for all stocks.


In [ ]:
# Sanity check on one stock - do the parsed datetimes actually look right?
check_stock = list(stock_data.keys())[0]
stock_data[check_stock][["Time Interval", "datetime", "Row_Type"]].head(15)


,Time Interval,datetime,Row_Type
0,Summary,NaT,Summary
1,01JAN2016_00:00:00.000000,NaT,Date_Header
2,07:30 - 07:45,2016-01-01 07:30:00,Interval
3,07:45 - 08:00,2016-01-01 07:45:00,Interval
4,08:00 - 08:15,2016-01-01 08:00:00,Interval
5,08:15 - 08:30,2016-01-01 08:15:00,Interval
6,08:30 - 08:45,2016-01-01 08:30:00,Interval
7,08:45 - 09:00,2016-01-01 08:45:00,Interval
8,09:00 - 09:15,2016-01-01 09:00:00,Interval
9,09:15 - 09:30,2016-01-01 09:15:00,Interval


In [ ]:
interval_data = {}

for stock, df in stock_data.items():
    interval_data[stock] = (
        df[df["Row_Type"] == "Interval"]
        .copy()
        .reset_index(drop=True)
    )

for stock, df in interval_data.items():
    print(f"{stock:25s} {len(df)} interval rows")


Eicher_Motors             74100 interval rows
Mahindra_and_Mahindra     74597 interval rows
HDFCB                     74590 interval rows
ICICIBC                   74690 interval rows
ONGC                      74702 interval rows
Reliance                  74701 interval rows
Hindustan_Unilever        74545 interval rows
ITC                       74688 interval rows
INFOSYS                   74694 interval rows
TCS                       74668 interval rows


Fixing the volume column

In [ ]:
sample_stock = list(interval_data.keys())[0]
print(interval_data[sample_stock]["Volume"].dtype)
interval_data[sample_stock]["Volume"].tail(10)


object


,Volume
74090,9700
74091,18833
74092,8111
74093,12125
74094,15757
74095,71059
74096,73632
74097,390
74098,12
74099,N.A.


In [ ]:
for stock, df in interval_data.items():
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce")

print("Rows where Volume was unparseable (e.g. was 'N.A.'):\n")
for stock, df in interval_data.items():
    bad = df["Volume"].isna().sum()
    print(f"{stock:25s} {bad} rows ({bad / len(df) * 100:.3f}%)")


Rows where Volume was unparseable (e.g. was 'N.A.'):

Eicher_Motors             1605 rows (2.166%)
Mahindra_and_Mahindra     1606 rows (2.153%)
HDFCB                     1606 rows (2.153%)
ICICIBC                   1606 rows (2.150%)
ONGC                      1607 rows (2.151%)
Reliance                  1607 rows (2.151%)
Hindustan_Unilever        1606 rows (2.154%)
ITC                       1606 rows (2.150%)
INFOSYS                   1606 rows (2.150%)
TCS                       1607 rows (2.152%)


In [ ]:
# rows with no trading activity
# to compute a feature or target from on a bar where zero shares traded, droped them.
for stock, df in interval_data.items():
    before = len(df)
    interval_data[stock] = df.dropna(subset=["Volume"]).reset_index(drop=True)
    dropped = before - len(interval_data[stock])
    if dropped:
        print(f"{stock}: dropped {dropped} rows with unusable Volume")


Eicher_Motors: dropped 1605 rows with unusable Volume
Mahindra_and_Mahindra: dropped 1606 rows with unusable Volume
HDFCB: dropped 1606 rows with unusable Volume
ICICIBC: dropped 1606 rows with unusable Volume
ONGC: dropped 1607 rows with unusable Volume
Reliance: dropped 1607 rows with unusable Volume
Hindustan_Unilever: dropped 1606 rows with unusable Volume
ITC: dropped 1606 rows with unusable Volume
INFOSYS: dropped 1606 rows with unusable Volume
TCS: dropped 1607 rows with unusable Volume


Auto-checking every stock for problems

In [ ]:
validation_rows = []

for stock, df in interval_data.items():
    validation_rows.append({
        "Stock": stock,
        "Rows": len(df),
        "Start": df["datetime"].min(),
        "End": df["datetime"].max(),
        "Duplicate_Timestamps": df["datetime"].duplicated().sum(),
        "In_Time_Order": df["datetime"].is_monotonic_increasing,
        "Nulls_OHLCV": df[["Open", "High", "Low", "Close", "Volume"]].isna().sum().sum(),
        "NonPositive_Prices": ((df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)).sum(),
    })

validation_df = pd.DataFrame(validation_rows)
validation_df


,Stock,Rows,Start,End,Duplicate_Timestamps,In_Time_Order,Nulls_OHLCV,NonPositive_Prices
0,Eicher_Motors,72495,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
1,Mahindra_and_Mahindra,72991,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
2,HDFCB,72984,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
3,ICICIBC,73084,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
4,ONGC,73095,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
5,Reliance,73094,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
6,Hindustan_Unilever,72939,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
7,ITC,73082,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
8,INFOSYS,73088,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0
9,TCS,73061,2016-01-01 07:30:00,2026-07-31 14:15:00,0,True,0,0


In [ ]:
stock_to_diagnose = "ONGC"

problem_df = interval_data[stock_to_diagnose]
print(f"{stock_to_diagnose} — date range covered by each source file:\n")
print(
    problem_df.groupby("Source_File")["datetime"]
    .agg(["min", "max", "count"])
    .sort_values("min")
)


ONGC — date range covered by each source file:

                                         min                 max  count
Source_File                                                            
ongc_15min_2016.csv      2016-01-01 07:30:00 2016-12-30 14:15:00   6890
ongc_15min_2017.csv      2017-01-02 07:30:00 2017-12-29 14:15:00   6899
ongc_15min_2018.csv      2018-01-01 07:30:00 2018-12-31 14:15:00   6866
ongc_15min_2019.csv      2019-01-01 07:30:00 2019-12-31 14:15:00   6838
ongc_15min_2020.csv      2020-01-01 07:30:00 2020-12-31 14:15:00   7031
ongc_15min_2021.csv      2021-01-01 07:30:00 2021-12-31 14:15:00   6913
ongc_15min_2022.csv      2022-01-03 07:30:00 2022-12-30 14:15:00   6923
ongc_15min_2023.csv      2023-01-02 07:30:00 2023-12-29 14:15:00   6867
ongc_15min_2024.csv      2024-01-01 07:30:00 2024-12-31 14:15:00   6913
ongc_15min_2025.csv      2025-01-01 07:30:00 2025-12-31 14:15:00   6951
ongc_15min_July_2026.csv 2026-01-01 07:30:00 2026-07-31 14:15:00   4004


In [ ]:
for stock, df in interval_data.items():
    before = len(df)
    df = df.drop_duplicates(subset=["datetime"], keep="first")
    df = df.sort_values("datetime").reset_index(drop=True)
    interval_data[stock] = df
    if before != len(df):
        print(f"{stock}: removed {before - len(df)} duplicate-timestamp rows")

print("\nAll stocks de-duplicated and time-sorted.")



All stocks de-duplicated and time-sorted.


In [ ]:
final_check_rows = []

for stock, df in interval_data.items():
    final_check_rows.append({
        "Stock": stock,
        "Rows": len(df),
        "Duplicate_Timestamps": df["datetime"].duplicated().sum(),
        "In_Time_Order": df["datetime"].is_monotonic_increasing,
        "Nulls_OHLCV": df[["Open", "High", "Low", "Close", "Volume"]].isna().sum().sum(),
    })

final_check_df = pd.DataFrame(final_check_rows)
final_check_df


,Stock,Rows,Duplicate_Timestamps,In_Time_Order,Nulls_OHLCV
0,Eicher_Motors,72495,0,True,0
1,Mahindra_and_Mahindra,72991,0,True,0
2,HDFCB,72984,0,True,0
3,ICICIBC,73084,0,True,0
4,ONGC,73095,0,True,0
5,Reliance,73094,0,True,0
6,Hindustan_Unilever,72939,0,True,0
7,ITC,73082,0,True,0
8,INFOSYS,73088,0,True,0
9,TCS,73061,0,True,0


Saving cleaned data

In [ ]:
FINAL_COLUMNS = {
    "datetime": "datetime",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume",
    "Tick Count": "tick_count",
}

cleaned_data = {}

for stock, df in interval_data.items():
    clean_df = df[list(FINAL_COLUMNS.keys())].rename(columns=FINAL_COLUMNS)
    clean_df = clean_df.sort_values("datetime").reset_index(drop=True)
    cleaned_data[stock] = clean_df

    safe_name = stock.replace(" ", "_").replace("&", "and")
    out_base = os.path.join(CLEAN_DATA_PATH, safe_name)
    clean_df.to_parquet(out_base + ".parquet", index=False)
    clean_df.to_csv(out_base + ".csv", index=False)

    print(f"{stock:25s} -> saved {len(clean_df)} rows")


Eicher_Motors             -> saved 72495 rows
Mahindra_and_Mahindra     -> saved 72991 rows
HDFCB                     -> saved 72984 rows
ICICIBC                   -> saved 73084 rows
ONGC                      -> saved 73095 rows
Reliance                  -> saved 73094 rows
Hindustan_Unilever        -> saved 72939 rows
ITC                       -> saved 73082 rows
INFOSYS                   -> saved 73088 rows
TCS                       -> saved 73061 rows


In [ ]:
inventory_path = os.path.join(RESULTS_PATH, "Cleaned_Data_Inventory.csv")
final_check_df.to_csv(inventory_path, index=False)
print("Saved:", inventory_path)


Saved: /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/04_Results/Cleaned_Data_Inventory.csv
